# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egeklc/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The chosen lane is Lane 2 — Refresh / Content Opportunity Scoring. The goal of this project is to prioritize which pages should be reviewed first. To achieve this, the model will rank the pages from highest to lowest score.

For this project, Logistic Regression and Random Forest will be used. Logistic Regression is a simple and interpretable model, while Random Forest can capture nonlinear relationships between features. The baseline, Logistic Regression model, and Random Forest model will be evaluated using the same data split and Precision@50 metric.

In [1]:
!git clone https://github.com/egeklc/flyrank-internship.git

import os
os.chdir("flyrank-internship")
print(os.getcwd())

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.
/content/flyrank-internship


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)


print("Number of pages: ", len(df))
print("Number of clients: ", df["client_id"].nunique())

Number of pages:  30000
Number of clients:  32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client grouped split will be used. Time-aware split cannot be used because the data contains one snapshot per content page. The split will be performed based on `client_id` to make sure that content from the clients does not appear in both training and test sets.

In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

target_col = "is_declining"
group_col = "client_id"

excluded_cols = ['content_id', 'client_id',
                 "trend_direction", "trend_pct", "is_declining",
                 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
                 'clicks_prev_30d', 'sessions_prev_30d'
                 ]

num_cols = [col for col in df.select_dtypes(include=["int64", "float64"]).columns
            if col not in excluded_cols
            ]

cat_cols = [col for col in df.select_dtypes(include=["object"]).columns
            if col not in excluded_cols
            ]

print(num_cols)
print(cat_cols)


feature_cols = num_cols + cat_cols
model_df = df[feature_cols + [target_col, group_col]].copy()

['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size = 0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df[group_col]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

In [5]:
print("Train rows:", len(X_train), " | Test rows:", len(X_test))
print("Number of Clients in Train Set:", model_df.iloc[train_idx]["client_id"].nunique())
print("Number of Clients in Test Set:", model_df.iloc[test_idx]["client_id"].nunique())

overlap = set(model_df.iloc[train_idx]["client_id"]) & set(model_df.iloc[test_idx]["client_id"])
print("Number of Overlapping Clients:", len(overlap))

Train rows: 22885  | Test rows: 7115
Number of Clients in Train Set: 24
Number of Clients in Test Set: 8
Number of Overlapping Clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression Model
numeric_transformer_log = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer_log = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_log = ColumnTransformer([
    ("num", numeric_transformer_log, num_cols),
    ("cat", categorical_transformer_log, cat_cols)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor_log),
    ("classifier", LogisticRegression(
        random_state=42,
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
    ))
])

In [7]:
# Random Forest Model
numeric_transformer_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0))
])

categorical_transformer_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_rf = ColumnTransformer([
    ("num", numeric_transformer_rf, num_cols),
    ("cat", categorical_transformer_rf, cat_cols)
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=15,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [8]:
logistic_model.fit(X_train, y_train)
logistic_scores = logistic_model.predict_proba(X_test)[:, 1]

random_forest_model.fit(X_train, y_train)
rf_scores = random_forest_model.predict_proba(X_test)[:, 1]


In [9]:
# Baseline Model

volume_floor_train = train_df[(train_df["avg_position"] > 0) &(train_df["impressions_90d"] >= 500)].copy()

expected_ctr_train = (volume_floor_train.groupby("position_tier")["ctr"]
                      .median()
                      .rename("expected_ctr")
                      )

max_impressions_train = train_df["impressions_90d"].max()

baseline_test = test_df.copy()

baseline_test["stale"] = (baseline_test["days_since_last_update"] >= 91).astype(int)
baseline_test["expected_ctr"] = (baseline_test["position_tier"].map(expected_ctr_train))
baseline_test["ctr_gap"] = (baseline_test["expected_ctr"] - baseline_test["ctr"]).clip(lower=0)
baseline_test.loc[baseline_test["avg_position"] <= 0, "ctr_gap"] = 0
baseline_test["baseline_score"] = (baseline_test["stale"]* (baseline_test["impressions_90d"] / max_impressions_train)* baseline_test["ctr_gap"])

In [10]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return y_true.iloc[top_k].mean()

In [11]:
from sklearn.metrics import roc_auc_score

baseline_p50 = precision_at_k(y_test, baseline_test["baseline_score"],k=50)
logistic_p50 = precision_at_k(y_test, logistic_scores, k=50)
rf_p50 = precision_at_k(y_test, rf_scores, k=50)

baseline_roc_auc = roc_auc_score(y_test, baseline_test["baseline_score"])
logistic_roc_auc = roc_auc_score(y_test, logistic_scores)
rf_roc_auc = roc_auc_score(y_test, rf_scores)


In [12]:
comparison_df = pd.DataFrame({"Model":["Baseline", "Logistic Regression", "Random Forest"],
                              "Precision@50":[baseline_p50, logistic_p50, rf_p50],
                              "ROC-AUC":[baseline_roc_auc, logistic_roc_auc, rf_roc_auc]
                              })
print(comparison_df)

                 Model  Precision@50   ROC-AUC
0             Baseline          0.50  0.504026
1  Logistic Regression          0.66  0.574678
2        Random Forest          0.68  0.600047


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The baseline model achieved a Precision@50 of 0.500 and ROC-AUC of 0.504, while Logistic Regression model achieved a Precision@50 of 0.660 and ROC-AUC of 0.575. The Random Forest model achieved a Precision@50 of 0.680 and ROC-AUC of 0.600.

Permutation importance shows that `days_with_impressions` is the strongest feature followed by `impressions_90d`, `avg_position`, and `clicks_90d`. The remaining features have smaller contributions.

In [13]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm_importance.importances_mean,
    "Std": perm_importance.importances_std
}).sort_values("Importance", ascending=False)

print(feature_importance.head(10).round(4).to_string(index=False))


              Feature  Importance    Std
days_with_impressions      0.0408 0.0033
      impressions_90d      0.0115 0.0006
         avg_position      0.0081 0.0011
           clicks_90d      0.0054 0.0001
                  ctr      0.0036 0.0006
        position_tier      0.0034 0.0013
   days_with_sessions      0.0024 0.0002
          scroll_rate      0.0022 0.0005
      impression_tier      0.0022 0.0004
         sessions_90d      0.0012 0.0002


In [14]:
error_analysis = df.iloc[test_idx].copy()

error_analysis["rf_score"] = rf_scores
error_analysis["actual"] = y_test.values
error_analysis["predicted_top50"] = 0
top_50_idx = np.argsort(rf_scores)[::-1][:50]
error_analysis.iloc[top_50_idx, error_analysis.columns.get_loc("predicted_top50")] = 1

In [15]:
false_positives = error_analysis[
    (error_analysis["predicted_top50"] == 1) & (error_analysis["actual"] == 0)
].sort_values("rf_score", ascending=False).head(3)

false_negatives = error_analysis[
    (error_analysis["predicted_top50"] == 0) & (error_analysis["actual"] == 1)
].sort_values("rf_score", ascending=True).head(3)

print("Top 3 False Positives:")
print(false_positives[
    ["content_id", "days_with_impressions", "impressions_90d", "avg_position",
     "clicks_90d", "ctr", "days_since_last_update", "rf_score"]
].to_string(index=False))

print(f"\n\n")
print("Top 3 False Negatives:")
print(false_negatives[
    ["content_id", "days_with_impressions", "impressions_90d", "avg_position",
     "clicks_90d", "ctr", "days_since_last_update", "rf_score"]
].to_string(index=False))

Top 3 False Positives:
          content_id  days_with_impressions  impressions_90d  avg_position  clicks_90d  ctr  days_since_last_update  rf_score
content_2ba626fea4d6                     58              360           7.2           0  0.0                     104  0.949247
content_8f1409b2674e                     66              209          20.0           0  0.0                     104  0.916716
content_41baf0722ad9                     88             3115          12.8           0  0.0                     104  0.912858



Top 3 False Negatives:
          content_id  days_with_impressions  impressions_90d  avg_position  clicks_90d  ctr  days_since_last_update  rf_score
content_16f38acf0f26                      2                2          50.0           0  0.0                      20  0.129695
content_e3a34c1706cf                      2                2           5.0           0  0.0                     104  0.134898
content_a4c38287770e                      2                2         

The false positives are mainly pages with zero clicks and zero CTR combined with relatively high visibility and longer periods since the last update. This suggests that the model can assign high decline scores to pages with weak observed performance, even when they are labeled as non-declining.

The false negatives have very limited search history, with only two days with impressions and two impressions in the 90-day period. Their low RF scores suggest that the model has difficulty identifying declining pages when there is insufficient performance history available.

## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.